In [1]:
import json
import folium 
from IPython.display import display
from folium.plugins import MarkerCluster
from collections import Counter
import os
from tqdm import tqdm

In [2]:
from utils_v2 import getAreaStats

In [3]:

def loadJSON(path):
    with open(path, "r") as file:
        file = json.load(file)
    return file


# Specify the path to your JSON file
frames_json = "/Users/bakuljangley/Documents/TUDThesis/mapillary_utils/zod/frames_v1.json"
drives_json = "/Users/bakuljangley/Documents/TUDThesis/mapillary_utils/zod/zod_drives.json"

zodFrames = loadJSON(frames_json)
print(f"Loaded {len(zodFrames)} frames.")


zodDrives = loadJSON(drives_json)
print(f"Loaded {len(zodDrives)} drives.")




Loaded 100000 frames.
Loaded 29 drives.


In [6]:
import json
import os
from tqdm import tqdm

def read_existing_data(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)
        return {item['frame_id']: item for item in data}
    return {}

def append_frame_stats(file_path, zodFrames, getAreaStats):
    existing_data = read_existing_data(file_path)
    
    # If file doesn't exist, create it with an empty list
    if not os.path.exists(file_path):
        with open(file_path, 'w') as f:
            json.dump([], f)
    
    with open(file_path, 'r+') as f:
        # Load existing data
        data = json.load(f)
        
        # Move to the end of the file, minus 1 character (to overwrite the closing bracket)
        f.seek(0, 2)
        f.seek(f.tell() - 1, 0)
        
        # If the file is not empty (i.e., it's not just []), add a comma
        if f.tell() > 1:
            f.write(',\n')
        first = True
        for frame_id, frame_data in tqdm(zodFrames.items(), desc="Processing frames"):
            if frame_id not in existing_data:
                # Process new frame
                lat = frame_data['lat']
                long = frame_data['long']
                num_photos, unique_sequences = getAreaStats([lat, long],x_dist=0.00005,y_dist=0.00005)

                frame_info = {
                    'frame_id': frame_id,
                    'latitude': lat,
                    'longitude': long,
                    'num_photos': num_photos,
                    'unique_sequences': unique_sequences
                }
                if not first:
                    f.write("\n")
                first = False
                # Append new frame info to the file
                json.dump(frame_info, f) #adds frame info to the json file
                f.write(',')
                
                # Flush the write buffer to ensure it's written to disk
                f.flush()
        
        # Remove the last comma and close the JSON array
        f.seek(f.tell() - 1, 0)
        f.write(']')
        f.truncate()

# Usage
file_path = 'zod/frame_stats_v4.json'
append_frame_stats(file_path, zodFrames, getAreaStats)


Processing frames:   2%|▏         | 1603/100000 [45:20<46:23:15,  1.70s/it] 


KeyboardInterrupt: 